In [ ]:
@njit
def calculate_score(data, target_col_idx, metric1_col_idx, metric2_col_idx, indices_col_idx):
    sorted_indices = np.argsort(data[:, indices_col_idx])
    sorted_data = data[sorted_indices]
    filter_indices = filter_trades_by_single_volume(sorted_data[:, target_col_idx], sorted_data[:, metric1_col_idx], sorted_data[:, metric2_col_idx])
    return sorted_data[np.array(filter_indices), target_col_idx].sum()


@njit
def calculate_score3(data):
    assert len(data) > 5
    mean_return = np.mean(data)
    std_return = np.std(data)
    return mean_return / std_return

def optimize_thresholds_njit(df, train_columns, target_column, min_samples, max_splits, tree_max_samples_calc_score):
    df['indices'] = range(len(df))
    
    target_col_idx = df.columns.get_loc(target_column)
    metric1_col_idx = df.columns.get_loc('metric1')
    metric2_col_idx = df.columns.get_loc('metric2')
    metric3_col_idx = df.columns.get_loc('metric3')
    metric4_col_idx = df.columns.get_loc('metric4')
    indices_col_idx = df.columns.get_loc('indices')
    col_mapping = list(range(df.shape[1]))
    assert df.shape[1] == len(train_columns) + 6

@njit
def filter_trades_by_single_volume(targets, starts, ends):
    previous_end = 0
    indexes = []
    for i in range(len(targets)):
        if previous_end < starts[i] or (i > 0 and starts[i] < starts[i-1]):
            indexes.append(i)
            previous_end = ends[i]
    return indexes

def calculate_statistics(df):
    mean_return = df.target.mean()
    std_return = df.target.std(ddof=1) + 0.000001
    sharpe_ratio = 0
    if len(df) > min_samples_to_calc_sharpe_ratio:
        sharpe_ratio = round((mean_return / std_return), 2) # Sharpe ratio
    return df.target.sum(), sharpe_ratio

def read_target(train_test_files, direction, profit, loss, time_in_ms, order_type):
    y_train = []
    y_test = []
    train_sub_rows = []
    test_sub_rows = []
    train_files = train_test_files[0]
    test_files = train_test_files[1]
    for idx in range(len(train_files)):
        df = pd.read_csv(f'target/{train_files[idx]}_p{profit}_l{loss}_d{direction}_t{time_in_ms}_o{order_type}.csv', sep=';', engine='pyarrow')
        df.reset_index(inplace=True, drop=True)
        df.rename(columns={df.columns[0]: 'target', df.columns[1]: 'start', df.columns[2]: 'end'}, inplace=True)
        train_sub_rows.append(filter_rows_with_no_deals(df))
        y_train.append(df)
    for idx in range(len(test_files)):
        df = pd.read_csv(f'target/{test_files[idx]}_p{profit}_l{loss}_d{direction}_t{time_in_ms}_o{order_type}.csv', sep=';', engine='pyarrow')
        df.reset_index(inplace=True, drop=True)
        df.rename(columns={df.columns[0]: 'target', df.columns[1]: 'start', df.columns[2]: 'end'}, inplace=True)
        test_sub_rows.append(filter_rows_with_no_deals(df))
        y_test.append(df)
    return y_train, y_test, train_sub_rows, test_sub_rows


def build_1d_plot(train_df, test_df, columns, target_column_name='target', metric_range=None):
    if metric_range is None:
        scatter1 = hv.Scatter(train_df, kdims=columns, vdims=[target_column_name]).opts(
            color=target_column_name, cmap='coolwarm', width=500, height=400)
        scatter2 = hv.Scatter(test_df, kdims=columns, vdims=[target_column_name]).opts(
            color=target_column_name, cmap='coolwarm', width=500, height=400)
        subplots = scatter1 + scatter2
    else:
        scatter1 = hv.Scatter(train_df, kdims=columns, vdims=[target_column_name]).opts(
            color=target_column_name, cmap='coolwarm', clim=metric_range, width=500, height=400)
        scatter2 = hv.Scatter(test_df, kdims=columns, vdims=[target_column_name]).opts(
            color=target_column_name, cmap='coolwarm', clim=metric_range, width=500, height=400)
        subplots = scatter1 + scatter2
    return subplots
        
        
def sample_from_df(df_filtered, sampling_size, columns):
    if sampling_enabled:
        indexes = sample_trades_by_single_volume_and_size(df_filtered.target.values, df_filtered.start.values, df_filtered.end.values, sampling_size)
        df_filtered = df_filtered.iloc[indexes]
    df_filtered['single_profit'] = df_filtered.target.values
    df_filtered['metric1'] = df_filtered.start.values
    df_filtered['metric2'] = df_filtered.end.values
    df_filtered['metric3'] = 0
    df_filtered['metric4'] = 0
    df_filtered.drop(['target', 'start', 'end'], axis=1, inplace=True)
    df_filtered.dropna(subset=columns, inplace=True)
    return df_filtered
    
def iterate(pld):
    hv.extension('plotly')
    merged_files = pld[0]
    train_columns = pld[1]
    plot_types = pld[2]
    print(train_columns)
    os.write(1, ('start ' + str(train_columns) + ' ' + str(time.time()) + '\n').encode())
    x_train, x_test, train_columns = read_train(merged_files, train_columns)
    for direction in directions:
        for profit in profits:
            for loss in losses:
                for time_in_ms in timers_in_ms:
                    for order_type in order_types:
                        y_train, y_test, train_sub_rows, test_sub_rows = read_target(merged_files, direction, profit, loss, time_in_ms, order_type)
                        merged_x_train, merged_y_train, merged_x_test, merged_y_test = merge_and_filter_datasets(x_train, x_test, y_train, y_test, train_sub_rows, test_sub_rows)
                        train_lst_for_stats, test_lst_for_stats = filter_datasets_for_stats(x_train, x_test, y_train, y_test, train_sub_rows, test_sub_rows)
                        
                        train_df = pd.concat([merged_x_train, merged_y_train], axis=1).copy()
                        test_df = pd.concat([merged_x_test, merged_y_test], axis=1).copy()
                        train_df_filtered = filter_df_by_condition(train_df)
                        test_df_filtered  = filter_df_by_condition(test_df)
                        train_df_sampled = sample_from_df(train_df_filtered, sampling_size_train, train_columns)
                        test_df_sampled = sample_from_df(test_df_filtered, sampling_size_test, train_columns)
                        train_df_plot = train_df_sampled[::max(len(train_df_sampled) // plot_rows_limit, 1)]
                        test_df_plot = test_df_sampled[::max(len(test_df_sampled) // plot_rows_limit, 1)]
                            
                        if plot_types == 1:
                            subplots = build_1d_plot(train_df_plot, test_df_plot, train_columns, target_column_name, plot_metric_range)
                            train_best_score, train_named_best_path = optimize_thresholds_njit(train_df_sampled, train_columns, target_column_name, tree_search_min_samples, tree_search_max_splits, tree_max_samples_calc_score)
                            test_best_score, train_single_sum, train_sharpe_ratio, test_single_sum, test_sharpe_ratio, len_train, let_single_train = apply_filters_and_metrics(train_df_sampled, test_df_sampled, train_named_best_path, train_lst_for_stats, test_lst_for_stats, train_best_score, train_columns)
                            save_to_file(merged_files, subplots, train_best_score, test_best_score, train_named_best_path, train_single_sum, train_sharpe_ratio, test_single_sum, test_sharpe_ratio, len_train, let_single_train, time_in_ms, profit, loss)
                        del y_train, y_test
                        del merged_x_train, merged_y_train, merged_x_test, merged_y_test
                        del train_lst_for_stats, test_lst_for_stats
                        del train_df, test_df, train_df_filtered, test_df_filtered
                        del train_df_sampled, test_df_sampled, train_df_plot, test_df_plot
                        gc.collect()
    os.write(1, ('finish ' + str(train_columns)  + ' ' + str(time.time()) + '\n').encode())

In [ ]:
# for single feature
path = 'D:\\Research\\Plots\\single plots'

# option1
# single_feat_filtered_columns = [[col] for col in filtered_columns]
# option2
single_feat_filtered_columns = [['st_price_bid_minus_shift']]

print(len(single_feat_filtered_columns))
chunk_size = 10
sublists = [single_feat_filtered_columns[i:i + chunk_size] for i in range(0, len(single_feat_filtered_columns), chunk_size)]
print('total chunks = ', len(sublists))
for idx, lst in enumerate(sublists):
    print('chunk ', idx)
    Parallel(n_jobs=8, verbose=10, backend='loky')(delayed(iterate)(pld) for pld in itertools.product(merged_files_combination, lst, [1]))
    gc.collect()

    # for pld in itertools.product(merged_files_combination, lst, [1]):
    #     iterate(pld)